# PLAIN DDP TEST

In [0]:
import os, socket, time, torch
import torch.distributed as dist
import torch.nn as nn, torch.optim as optim
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader, DistributedSampler

class ToyDataset(Dataset):
    def __init__(self, n=50_000, in_dim=784, n_classes=10):
        g = torch.Generator().manual_seed(0)
        self.x = torch.randn(n, in_dim, generator=g)
        self.y = torch.randint(0, n_classes, (n,), generator=g)
    def __len__(self): return self.x.size(0)
    def __getitem__(self, i): return self.x[i], self.y[i]

class MLP(nn.Module):
    def __init__(self, in_dim=784, hidden=512, n_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, n_classes)
        )
    def forward(self, x): return self.net(x)

def train_main(epochs=3, batch_size=128, lr=1e-3):
    # --- DDP setup ---
    rank        = int(os.environ["RANK"])
    world_size  = int(os.environ["WORLD_SIZE"])
    local_rank  = int(os.environ["LOCAL_RANK"])
    hostname    = socket.gethostname()

    torch.cuda.set_device(local_rank)
    dist.init_process_group(backend="nccl", init_method="env://",
                            world_size=world_size, rank=rank)

    if rank == 0:
        print(f"🌍 World size = {world_size}")
    print(f"Rank {rank:02d}/{world_size} on host {hostname} using GPU {local_rank}")

    # --- Data / Model setup ---
    dataset = ToyDataset()
    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank, shuffle=True)
    loader  = DataLoader(dataset, batch_size=batch_size, sampler=sampler,
                         num_workers=2, pin_memory=True)

    model = MLP().to(local_rank)
    model = DDP(model, device_ids=[local_rank])
    opt   = optim.AdamW(model.parameters(), lr=lr)
    lossf = nn.CrossEntropyLoss()

    dist.barrier()  # wait all ranks before timing
    if rank == 0: print("All ranks synchronized. Starting training...\n")

    # --- Training loop ---
    for epoch in range(epochs):
        sampler.set_epoch(epoch)
        model.train()
        start_t = time.time()
        running = 0.0
        for i, (x, y) in enumerate(loader):
            x, y = x.to(local_rank, non_blocking=True), y.to(local_rank, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            loss = lossf(model(x), y)
            loss.backward()
            opt.step()
            running += loss.item()

            if (i + 1) % 50 == 0:
                avg_loss = running / 50
                if rank == 0:
                    print(f"[Epoch {epoch+1}/{epochs}] Step {i+1:04d}  Loss={avg_loss:.4f}")
                running = 0.0

        elapsed = time.time() - start_t
        if rank == 0:
            print(f"Epoch {epoch+1} done in {elapsed:.1f}s ({len(loader)/elapsed:.1f} it/s)\n")

    dist.destroy_process_group()
    if rank == 0:
        print("Training complete.")

In [0]:
from pyspark.ml.torch.distributor import TorchDistributor

# 4 nodes × 4 GPUs per node = 16 processes
result = TorchDistributor(
    num_processes=16,     # total processes == total GPUs
    local_mode=False,     # multi-node
    use_gpu=True
).run(train_main, epochs=2, batch_size=128, lr=1e-3)